# M16 · 07 — Evaluation: the §10 functional gates on the frozen test split

`python -m llm.evaluate` runs the **app's own pipelines** (chapter titles, show notes, zero-shot
relevance) on the 49 test episodes of the task corpus (two held-out channels, `poco-se-habla` and
`lenny-s-podcast`, plus the 0024 test episodes). The transcripts are the corpus ones (Whisper),
and none of these episodes were used in training.

- `own-sft`: our model after task SFT. `own-dpo`: after DPO.
- `qwen`: Qwen3.5-9B relevance, the reference. Its two passes give Qwen's pass-to-pass Spearman.

Gates (selectable): titles ≤ 10 % mT5 fallback and ≥ 95 % right language; show notes ≥ 95 % valid
within 3 attempts and ≥ 95 % right language; relevance 100 % valid batches and Spearman with Qwen
≥ 0.8 × Qwen's pass-to-pass Spearman. Failing a gate does not hide the model in the app (0025
amendment): it is reported here and at C1.

In [1]:
import json, statistics
from pathlib import Path
from llm import HOME
RUNS, SFT, EVAL = HOME / 'runs', HOME / 'data' / 'sft', HOME / 'eval'
def load(p): return json.loads(Path(p).read_text())
def metrics(run): return [json.loads(l) for l in (RUNS / run / 'metrics.jsonl').read_text().splitlines()]
from llm.evaluate import gates, spearman, mean_passes

In [2]:
rows = {}
for label in ('own-sft', 'own-dpo'):
    if (EVAL / label).exists():
        rows[label] = gates(label)
        print(json.dumps(rows[label], indent=1))

{
 "label": "own-sft",
 "episodes": 49,
 "measured": "2026-11-19 00:45:07",
 "titles": {
  "chapters": 257,
  "fallback_rate": 0.0,
  "lang_ok_rate": 0.9961,
  "pass": true
 },
 "shownotes": {
  "episodes": 49,
  "valid_rate": 0.5714,
  "lang_ok_rate": 0.5714,
  "mean_reduce_attempts": 2.67,
  "pass": false
 },
 "relevance": {
  "batches": 1031,
  "failed": 0,
  "valid_rate": 1.0,
  "episodes_compared": 49,
  "spearman_vs_qwen": 0.2112,
  "qwen_pass_to_pass": 0.4903,
  "pass": false
 }
}
{
 "label": "own-dpo",
 "episodes": 49,
 "measured": "2026-11-19 00:45:08",
 "titles": {
  "chapters": 257,
  "fallback_rate": 0.0,
  "lang_ok_rate": 0.9844,
  "pass": true
 },
 "shownotes": {
  "episodes": 49,
  "valid_rate": 0.8571,
  "lang_ok_rate": 0.8571,
  "mean_reduce_attempts": 2.43,
  "pass": false
 },
 "relevance": {
  "batches": 1031,
  "failed": 0,
  "valid_rate": 1.0,
  "episodes_compared": 49,
  "spearman_vs_qwen": -0.0728,
  "qwen_pass_to_pass": 0.4903,
  "pass": false
 }
}


### Per channel

In [3]:
def per_channel(label):
    out = {}
    for p in sorted((EVAL / label).glob('*.json')):
        if p.name in ('run.json', 'gates.json'): continue
        e = load(p); c = out.setdefault(e['channel'], {'eps': 0, 'chapters': 0, 'fb': 0, 'sn_ok': 0, 'rho': []})
        c['eps'] += 1
        c['chapters'] += len(e['titles']['chapters']); c['fb'] += sum(x['fallback'] for x in e['titles']['chapters'])
        c['sn_ok'] += e['shownotes']['ok']
        q = EVAL / 'qwen' / p.name
        if q.exists():
            r = spearman(mean_passes(e['relevance']['passes']), mean_passes(load(q)['relevance']['passes']))
            if r is not None: c['rho'].append(r)
    for ch, c in out.items():
        print(f"{label:8s} {ch:22s} eps {c['eps']:2d} · title fallback {c['fb']}/{c['chapters']} · "
              f"show notes valid {c['sn_ok']}/{c['eps']} · relevance rho vs Qwen "
              f"{statistics.mean(c['rho']) if c['rho'] else float('nan'):.3f}")
for label in rows: per_channel(label)

own-sft  lenny-s-podcast        eps 17 · title fallback 0/80 · show notes valid 6/17 · relevance rho vs Qwen 0.407
own-sft  poco-se-habla          eps 26 · title fallback 0/153 · show notes valid 18/26 · relevance rho vs Qwen 0.101
own-sft  sastre-y-maldonado     eps  6 · title fallback 0/24 · show notes valid 4/6 · relevance rho vs Qwen 0.131
own-dpo  lenny-s-podcast        eps 17 · title fallback 0/80 · show notes valid 14/17 · relevance rho vs Qwen -0.073
own-dpo  poco-se-habla          eps 26 · title fallback 0/153 · show notes valid 23/26 · relevance rho vs Qwen -0.080
own-dpo  sastre-y-maldonado     eps  6 · title fallback 0/24 · show notes valid 5/6 · relevance rho vs Qwen -0.040


### Why show notes failed

In [4]:
from collections import Counter
import re
for label in rows:
    errs = Counter()
    for p in (EVAL / label).glob('*.json'):
        if p.name in ('run.json', 'gates.json'): continue
        s = load(p)['shownotes']
        if not s['ok']:
            inner = s['error'].split(' attempts (', 1)[-1].rstrip(')')
            for part in inner.split('; '):
                errs[re.sub(r'\d+', '#', part)[:80]] += 1
    print(label, dict(errs.most_common()))

own-sft {'# usable tags, need #': 32, '# usable titles, need #': 23, 'not valid JSON (Unterminated string starting at': 6, 'not valid JSON (Unterminated string starting at)': 1, 'not valid JSON (Invalid control character at': 1}
own-dpo {'# usable tags, need #': 12, '# usable titles, need #': 5, 'not valid JSON (Invalid control character at': 2, 'not valid JSON (Unterminated string starting at)': 2}


### Side by side: one episode per test channel

In [5]:
for label in rows:
    seen = set()
    for p in sorted((EVAL / label).glob('*.json')):
        if p.name in ('run.json', 'gates.json'): continue
        e = load(p)
        if e['channel'] in seen: continue
        seen.add(e['channel'])
        print(f"== {label} · {e['channel']} · {e['id']} ({e['lang']})")
        for c in e['titles']['chapters'][:6]: print('   title:', c['title'])
        s = e['shownotes']
        print('   show notes:', 'OK' if s['ok'] else 'FAILED', json.dumps(s.get('draft', {}).get('title_suggestions'), ensure_ascii=False))
        if s['ok']: print('   summary:', s['draft']['summary'][:300])

== own-sft · lenny-s-podcast · -ciSTkEVy30 (en)
   title: Shipping imperfect solutions to the wild and imperfect solution
   title: Machine learning and the DMable PM for product leaders
   title: Voice and voice changes the game
   show notes: FAILED null
== own-sft · poco-se-habla · 10JkMJqtmfw (es)
   title: Hotel de lujo en Miami y el hotel de lujo
   title: Holded Wallet y Holded Wallet para gestionar las finanzas
   title: El móvil fijo de la casa y el móvil de
   title: La amistad con alguien que se ha roto la amistad
   show notes: FAILED null
== own-sft · sastre-y-maldonado · DEf3O1LJ6HE (es)
   title: El hablante y el abusón de clase
   title: Lucas y el "Bombidge" de Lucas
   title: Los hombres que fuman menos y se fuman menos
   title: Esquema romántico de películas y casos de enamoramiento
   title: El último libro de Eduardo Mendoza y la verdad sobre
   show notes: OK ["El hablante y el abusón: El abusón de clase y la salud pública", "El hablante y el abuso de clase: El a

### Qwen's chapter titles for the same episodes (from the corpus run), for comparison

In [6]:
from llm.corpus import CORPUS
for p in sorted((EVAL / 'own-sft').glob('*.json'))[:4]:
    if p.name in ('run.json', 'gates.json'): continue
    e = load(p); q = load(CORPUS / 'processed' / e['id'] / 'chapters.json')
    print(f"== {e['channel']} {e['id']}")
    for a, b in zip(e['titles']['chapters'], q): print(f"   own : {a['title']}\n   qwen: {b['title']}")

== lenny-s-podcast -ciSTkEVy30
   own : Shipping imperfect solutions to the wild and imperfect solution
   qwen: Shipping imperfect AI products to iterate quickly
   own : Machine learning and the DMable PM for product leaders
   qwen: Onboarding, Privacy, and Direct User Relationships
   own : Voice and voice changes the game
   qwen: Predicting the biggest product trends for 2027
== poco-se-habla 10JkMJqtmfw
   own : Hotel de lujo en Miami y el hotel de lujo
   qwen: Summer Edition en Port Benidorm
   own : Holded Wallet y Holded Wallet para gestionar las finanzas
   qwen: Ana Abrito y Cristina hablan de Holded en Benidorm
   own : El móvil fijo de la casa y el móvil de
   qwen: Nostalgia de los antiguos amigos y padres
   own : La amistad con alguien que se ha roto la amistad
   qwen: ¿Se puede recuperar una amistad rota?
== lenny-s-podcast 19qrLchbQEc
   own : Selling to SpaceX and the Hail Mary approach
   qwen: Sales Pull and the Hail Mary Approach
   own : Selling to Startups an

## Conclusions (measured 2026-11-19, 49 test episodes)

| gate | own-sft | own-dpo | threshold |
|---|---|---|---|
| titles: mT5 fallback | 0 % | 0 % | ≤ 10 % |
| titles: right language | 99.6 % | 98.4 % | ≥ 95 % |
| show notes: valid in ≤ 3 attempts | 57 % | **86 %** | ≥ 95 % |
| relevance: valid batches | 100 % | 100 % | 100 % |
| relevance: Spearman vs Qwen | 0.211 | −0.073 | ≥ 0.8 × 0.490 = 0.392 |

- **Titles pass** for both. They are in the right language and on topic, but repetitive
  ("Holded Wallet y Holded Wallet…") and less specific than Qwen's.
- **Show notes fail the gate.** DPO took them from 57 % to 86 % valid; what still fails is too few
  distinct tags or titles, and a few truncated JSON strings.
- **Relevance fails, and DPO made it worse.** Both models emit smooth, content-independent shapes
  (ramps, arches) learnt from the form of Qwen's score lists, not from the blocks' content. The SFT
  model correlates with Qwen only on the English channel (0.41 on lenny-s-podcast).
- Qwen's own pass-to-pass Spearman is only 0.49, so the reference is noisy too.

**Installed in the app: `yts-own-110m-dpo`** (`models/llm/current`). It is the better of the two on
the tool that differs (show notes) and ties on titles; relevance fails with either. Per the 0025
amendment the model stays selectable despite the failed gates, and Qwen stays the default.